In [1]:
%pip install streamlit pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
%%writefile fraud_realtime_investigation.py

import joblib
import numpy as np
import pandas as pd
import streamlit as st

from datetime import datetime
from pathlib import Path

st.set_page_config(
    page_title="Real-Time Fraud Investigation",
    page_icon="🛡️",
    layout="wide"
)

BASE_FOLDER = Path(__file__).resolve().parent

MODEL_FILE = (
    BASE_FOLDER /
    "final_fraud_detection_pipeline.pkl"
)

FEATURE_FILE = (
    BASE_FOLDER /
    "final_model_feature_columns.pkl"
)

THRESHOLD_FILE = (
    BASE_FOLDER /
    "final_fraud_threshold.pkl"
)

TRANSACTION_FILE = (
    BASE_FOLDER /
    "kafka_transaction_results.csv"
)

FRAUD_ALERT_FILE = (
    BASE_FOLDER /
    "kafka_fraud_alerts.csv"
)

MONITORING_FILE = (
    BASE_FOLDER /
    "kafka_monitoring_alerts.csv"
)

ALERT_HISTORY_FILE = (
    BASE_FOLDER /
    "fraud_alert_history.csv"
)


def read_csv_safely(file_path):

    if not file_path.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(file_path)

    except pd.errors.EmptyDataError:
        return pd.DataFrame()

    except Exception as error:

        st.warning(
            f"Could not read {file_path.name}: "
            f"{error}"
        )

        return pd.DataFrame()


@st.cache_resource
def load_model_files():

    if not MODEL_FILE.exists():
        raise FileNotFoundError(
            f"Model file not found: {MODEL_FILE}"
        )

    if not FEATURE_FILE.exists():
        raise FileNotFoundError(
            f"Feature file not found: {FEATURE_FILE}"
        )

    model = joblib.load(MODEL_FILE)
    features = joblib.load(FEATURE_FILE)

    if THRESHOLD_FILE.exists():
        threshold = float(
            joblib.load(THRESHOLD_FILE)
        )
    else:
        threshold = 0.50

    return model, features, threshold


@st.cache_data(ttl=5)
def load_monitoring_files():

    transactions = read_csv_safely(
        TRANSACTION_FILE
    )

    fraud_alerts = read_csv_safely(
        FRAUD_ALERT_FILE
    )

    monitoring = read_csv_safely(
        MONITORING_FILE
    )

    history = read_csv_safely(
        ALERT_HISTORY_FILE
    )

    return (
        transactions,
        fraud_alerts,
        monitoring,
        history
    )


try:

    final_model, feature_columns, fraud_threshold = (
        load_model_files()
    )

    model_available = True

except Exception as error:

    model_available = False
    final_model = None
    feature_columns = []
    fraud_threshold = 0.50

    st.error(str(error))


def calculate_risk_details(
    fraud_probability,
    threshold
):

    risk_score = round(
        fraud_probability * 100,
        2
    )

    critical_threshold = max(
        0.80,
        threshold
    )

    if fraud_probability >= critical_threshold:

        risk_level = "Critical"
        action = "Block transaction and investigate"

    elif fraud_probability >= threshold:

        risk_level = "High"
        action = "Hold transaction for investigation"

    elif fraud_probability >= 0.30:

        risk_level = "Medium"
        action = "Monitor and verify transaction"

    else:

        risk_level = "Low"
        action = "Approve transaction"

    predicted_fraud = int(
        fraud_probability >= threshold
    )

    return (
        risk_score,
        risk_level,
        action,
        predicted_fraud
    )


def create_investigation_reasons(
    transaction,
    fraud_probability
):

    reasons = []

    amount = transaction.get(
        "amount",
        0
    )

    transaction_hour = transaction.get(
        "transaction_hour",
        12
    )

    failed_attempts = transaction.get(
        "failed_payment_attempts",
        0
    )

    login_attempts = transaction.get(
        "login_attempts",
        0
    )

    is_international = transaction.get(
        "is_international",
        0
    )

    new_device = transaction.get(
        "new_device",
        0
    )

    new_location = transaction.get(
        "new_location",
        0
    )

    if amount >= 5000:
        reasons.append(
            "Unusually high transaction amount"
        )

    if transaction_hour <= 4:
        reasons.append(
            "Transaction occurred at an unusual hour"
        )

    if failed_attempts >= 3:
        reasons.append(
            "Multiple failed payment attempts"
        )

    if login_attempts >= 5:
        reasons.append(
            "Multiple login attempts"
        )

    if is_international == 1:
        reasons.append(
            "International transaction"
        )

    if new_device == 1:
        reasons.append(
            "Transaction from a new device"
        )

    if new_location == 1:
        reasons.append(
            "Transaction from a new location"
        )

    if fraud_probability >= fraud_threshold:
        reasons.append(
            "Model probability crossed fraud threshold"
        )

    if not reasons:
        reasons.append(
            "No major rule-based indicator detected"
        )

    return reasons


def create_empty_transaction():

    transaction = {}

    for column in feature_columns:
        transaction[column] = np.nan

    return transaction

st.sidebar.title(
    "🛡️ Fraud Investigation"
)

selected_page = st.sidebar.radio(
    "Select Page",
    [
        "Real-Time Prediction",
        "Streaming Monitor",
        "Investigation Queue",
        "Alert History"
    ]
)

st.sidebar.markdown("---")

st.sidebar.metric(
    "Fraud Threshold",
    f"{fraud_threshold:.4f}"
)

if st.sidebar.button(
    "Refresh Data",
    use_container_width=True
):

    st.cache_data.clear()
    st.rerun()

if selected_page == "Real-Time Prediction":

    st.title(
        "Real-Time Transaction Prediction"
    )

    st.write(
        "Enter transaction information to calculate "
        "fraud probability, risk score and action."
    )

    if not model_available:

        st.stop()

    with st.form(
        "transaction_prediction_form"
    ):

        column1, column2, column3 = (
            st.columns(3)
        )

        with column1:

            amount = st.number_input(
                "Transaction Amount",
                min_value=0.0,
                value=500.0,
                step=100.0
            )

            transaction_hour = st.number_input(
                "Transaction Hour",
                min_value=0,
                max_value=23,
                value=12
            )

            transaction_day = st.number_input(
                "Transaction Day",
                min_value=1,
                max_value=31,
                value=15
            )

            transaction_month = st.number_input(
                "Transaction Month",
                min_value=1,
                max_value=12,
                value=8
            )

        with column2:

            existing_risk_score = st.number_input(
                "Existing Risk Indicator",
                min_value=0.0,
                max_value=100.0,
                value=20.0
            )

            transactions_last_24h = st.number_input(
                "Transactions in Last 24 Hours",
                min_value=0,
                value=1
            )

            failed_payment_attempts = (
                st.number_input(
                    "Failed Payment Attempts",
                    min_value=0,
                    value=0
                )
            )

            login_attempts = st.number_input(
                "Login Attempts",
                min_value=0,
                value=1
            )

        with column3:

            account_age_days = st.number_input(
                "Account Age in Days",
                min_value=0,
                value=365
            )

            is_international = st.selectbox(
                "International Transaction",
                options=[0, 1],
                format_func=lambda value:
                "Yes" if value == 1 else "No"
            )

            new_device = st.selectbox(
                "New Device",
                options=[0, 1],
                format_func=lambda value:
                "Yes" if value == 1 else "No"
            )

            new_location = st.selectbox(
                "New Location",
                options=[0, 1],
                format_func=lambda value:
                "Yes" if value == 1 else "No"
            )

        predict_button = st.form_submit_button(
            "Predict Fraud Risk",
            use_container_width=True
        )

    if predict_button:

        transaction = create_empty_transaction()

        input_values = {
            "amount": amount,
            "order_amount": amount,
            "transaction_hour": transaction_hour,
            "transaction_day": transaction_day,
            "transaction_month": transaction_month,
            "risk_score": existing_risk_score,
            "transactions_last_24h":
                transactions_last_24h,
            "failed_payment_attempts":
                failed_payment_attempts,
            "login_attempts": login_attempts,
            "account_age_days": account_age_days,
            "is_international": is_international,
            "new_device": new_device,
            "new_location": new_location
        }

        for column, value in input_values.items():

            if column in transaction:
                transaction[column] = value

        transaction_frame = pd.DataFrame(
            [transaction]
        )

        try:

            fraud_probability = float(
                final_model.predict_proba(
                    transaction_frame[
                        feature_columns
                    ]
                )[0, 1]
            )

            (
                model_risk_score,
                risk_level,
                recommended_action,
                predicted_fraud
            ) = calculate_risk_details(
                fraud_probability,
                fraud_threshold
            )

            case_id = (
                "CASE-"
                + datetime.now().strftime(
                    "%Y%m%d%H%M%S"
                )
            )

            result_column1, result_column2, result_column3 = (
                st.columns(3)
            )

            result_column1.metric(
                "Fraud Probability",
                f"{fraud_probability:.4f}"
            )

            result_column2.metric(
                "Model Risk Score",
                f"{model_risk_score:.2f}"
            )

            result_column3.metric(
                "Risk Level",
                risk_level
            )

            if risk_level == "Critical":
                st.error(
                    recommended_action
                )

            elif risk_level == "High":
                st.warning(
                    recommended_action
                )

            elif risk_level == "Medium":
                st.info(
                    recommended_action
                )

            else:
                st.success(
                    recommended_action
                )

            reasons = create_investigation_reasons(
                transaction,
                fraud_probability
            )

            investigation_result = pd.DataFrame({
                "case_id": [case_id],
                "created_at": [
                    datetime.now().isoformat()
                ],
                "fraud_probability": [
                    fraud_probability
                ],
                "risk_score": [
                    model_risk_score
                ],
                "risk_level": [
                    risk_level
                ],
                "predicted_fraud": [
                    predicted_fraud
                ],
                "recommended_action": [
                    recommended_action
                ],
                "investigation_reasons": [
                    "; ".join(reasons)
                ]
            })

            st.subheader(
                "Investigation Details"
            )

            st.dataframe(
                investigation_result,
                use_container_width=True,
                hide_index=True
            )

            st.download_button(
                "Download Investigation Report",
                investigation_result.to_csv(
                    index=False
                ),
                f"{case_id}.csv",
                "text/csv"
            )

        except Exception as error:

            st.error(
                f"Prediction failed: {error}"
            )


# ==================================================
# Streaming monitor
# ==================================================

elif selected_page == "Streaming Monitor":

    st.title(
        "Kafka Streaming Transaction Monitor"
    )

    (
        transaction_data,
        fraud_alerts,
        monitoring_alerts,
        alert_history
    ) = load_monitoring_files()

    if transaction_data.empty:

        st.warning(
            "No Kafka results are available. "
            "Run the producer and consumer first."
        )

    else:

        numeric_columns = [
            "risk_score",
            "fraud_probability",
            "predicted_fraud"
        ]

        for column in numeric_columns:

            if column in transaction_data.columns:

                transaction_data[column] = (
                    pd.to_numeric(
                        transaction_data[column],
                        errors="coerce"
                    )
                )

        total_transactions = len(
            transaction_data
        )

        total_fraud = int(
            transaction_data[
                "predicted_fraud"
            ].fillna(0).sum()
        )

        monitoring_count = int(
            transaction_data[
                "risk_level"
            ]
            .isin(
                [
                    "Medium",
                    "High",
                    "Critical"
                ]
            )
            .sum()
        )

        average_risk = (
            transaction_data[
                "risk_score"
            ].mean()
        )

        column1, column2, column3, column4 = (
            st.columns(4)
        )

        column1.metric(
            "Transactions",
            total_transactions
        )

        column2.metric(
            "Fraud Alerts",
            total_fraud
        )

        column3.metric(
            "Monitoring",
            monitoring_count
        )

        column4.metric(
            "Average Risk",
            f"{average_risk:.2f}"
        )

        if "risk_level" in transaction_data.columns:

            risk_counts = (
                transaction_data[
                    "risk_level"
                ]
                .value_counts()
                .reindex(
                    [
                        "Critical",
                        "High",
                        "Medium",
                        "Low"
                    ],
                    fill_value=0
                )
            )

            st.subheader(
                "Risk-Level Distribution"
            )

            st.bar_chart(
                risk_counts
            )

        st.subheader(
            "Latest Transactions"
        )

        latest_transactions = (
            transaction_data.tail(20)
        )

        st.dataframe(
            latest_transactions,
            use_container_width=True,
            hide_index=True
        )

elif selected_page == "Investigation Queue":

    st.title(
        "Fraud Investigation Queue"
    )

    (
        transaction_data,
        fraud_alerts,
        monitoring_alerts,
        alert_history
    ) = load_monitoring_files()

    queue_data = pd.concat(
        [
            fraud_alerts,
            monitoring_alerts
        ],
        ignore_index=True
    )

    if queue_data.empty:

        st.info(
            "No investigation cases are available."
        )

    else:

        if "stream_id" in queue_data.columns:

            queue_data = queue_data.drop_duplicates(
                subset=["stream_id"],
                keep="first"
            )

        if "risk_score" in queue_data.columns:

            queue_data["risk_score"] = (
                pd.to_numeric(
                    queue_data["risk_score"],
                    errors="coerce"
                )
            )

            queue_data = queue_data.sort_values(
                by="risk_score",
                ascending=False
            )

        queue_data = queue_data.reset_index(
            drop=True
        )

        selected_risk = st.multiselect(
            "Filter Investigation Risk",
            options=[
                "Critical",
                "High",
                "Medium"
            ],
            default=[
                "Critical",
                "High",
                "Medium"
            ]
        )

        if "risk_level" in queue_data.columns:

            queue_data = queue_data[
                queue_data["risk_level"].isin(
                    selected_risk
                )
            ]

        st.metric(
            "Open Investigation Cases",
            len(queue_data)
        )

        st.dataframe(
            queue_data,
            use_container_width=True,
            hide_index=True
        )

        st.download_button(
            "Download Investigation Queue",
            queue_data.to_csv(
                index=False
            ),
            "fraud_investigation_queue.csv",
            "text/csv"
        )

elif selected_page == "Alert History":

    st.title(
        "Fraud Alert and Notification History"
    )

    (
        transaction_data,
        fraud_alerts,
        monitoring_alerts,
        alert_history
    ) = load_monitoring_files()

    if alert_history.empty:

        st.info(
            "No notification history is available."
        )

    else:

        column1, column2 = st.columns(2)

        column1.metric(
            "Total Notifications",
            len(alert_history)
        )

        unique_transactions = (
            alert_history["stream_id"].nunique()
            if "stream_id"
            in alert_history.columns
            else len(alert_history)
        )

        column2.metric(
            "Unique Transactions",
            unique_transactions
        )

        st.dataframe(
            alert_history,
            use_container_width=True,
            hide_index=True
        )

st.markdown("---")

st.caption(
    "Real-Time Fraud Prediction, Risk Scoring "
    "and Transaction Investigation System"
)

Writing fraud_realtime_investigation.py


In [3]:
import subprocess
import sys

syntax_check = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        "fraud_realtime_investigation.py"
    ],
    capture_output=True,
    text=True
)

if syntax_check.returncode == 0:
    print("No syntax or indentation errors!")
else:
    print(syntax_check.stderr)

No syntax or indentation errors!


In [4]:
import subprocess
import sys

app_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "fraud_realtime_investigation.py"
    ]
)

print("Open: http://localhost:8501")

Open: http://localhost:8501


In [6]:
#to stop it later
app_process.terminate()